Ten skrypt wykorzystuje dwa lokalne modele językowe  do tworzenia, analizowania i ulepszania żartów typu "dad joke" poprzez współpracę trzech wyspecjalizowanych agentów: kreatora, parsera, i sędziego. System generuje żart, szczegółowo go analizuje pod kątem ośmiu kryteriów oceny, prezentuje wyniki w estetycznej tabeli, a następnie tworzy ulepszoną wersję na podstawie wcześniejszej krytyki.

# Setup

In [ ]:
!uv pip install colab-xterm
%load_ext colabxterm

In [ ]:
# Run in terminal
# curl -sSL https://ollama.ai/install.sh | sh && ollama serve

In [ ]:
%xterm

In [ ]:
!curl http://localhost:11434/api/pull -d '{  "model": "deepseek-r1:14b" }'

In [ ]:
!curl http://localhost:11434/api/pull -d '{  "model": "qwen2.5" }'

In [1]:
!pip install -q pydantic_ai pydantic rich

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.6/156.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.0/264.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.5/127.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.5/301.5 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

*   **pydantic_ai:** Biblioteka ta rozszerza możliwości Pydantic (biblioteki do walidacji danych) o integrację z modelami językowymi AI. Umożliwia ona wykorzystanie modeli AI do generowania i przetwarzania danych zgodnych ze schematami Pydantic.
*   **pydantic:** Jest to biblioteka służąca do definiowania typów danych i walidacji danych w Pythonie. Pomaga zapewnić, że dane spełniają określone kryteria przed ich użyciem.
*   **rich:** Biblioteka ta dodaje kolorowe i stylowe formatowanie tekstu do konsoli Pythona. Ułatwia to czytanie i interpretację wyników oraz komunikatów w terminalu.

In [ ]:
class CFG:
    model1 = "deepseek-r1:14b"
    model2 = "qwen2.5"

In [ ]:
# Standard library imports
from typing import List

# Third-party imports
from pydantic import BaseModel, Field, confloat
from rich import box
from rich.console import Console
from rich.table import Table

# Pydantic AI imports
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

*   **Third-party imports:** Zawiera importy z zewnętrznych bibliotek zainstalowanych za pomocą `pip`, takich jak:
    *   `pydantic`: Do definiowania modeli danych i walidacji. Używane są klasy `BaseModel`, `Field` oraz `confloat`.
    *   `rich`: Do tworzenia estetycznych interfejsów w konsoli, z wykorzystaniem klas `Console` i `Table`.  Używana jest również stała `box` do definiowania stylów ramek tabel.
*   **Pydantic AI imports:** Zawiera importy specyficzne dla biblioteki `pydantic_ai`, które umożliwiają integrację modeli językowych z Pydantic. Importowane są:
    *   `Agent`: Klasa reprezentująca agenta AI, który wykorzystuje model językowy do wykonywania zadań.
    *   `RunContext`:  Klasa przechowująca kontekst wykonania agenta.
    *   `OpenAIModel`: Model reprezentujący modele OpenAI dostępne w `pydantic_ai`.
    *   `OpenAIProvider`: Dostawca usług OpenAI dla `pydantic_ai`, umożliwiający komunikację z API OpenAI.

In [ ]:
# DeepSeekR1
deepseek_reasoner_model = OpenAIModel(
    CFG.model1, provider=OpenAIProvider(base_url="http://localhost:11434/v1")
)

deepseek_parser_model = OpenAIModel(
    CFG.model2, provider=OpenAIProvider(base_url="http://localhost:11434/v1")
)

Ten kod tworzy dwie instancje klasy `OpenAIModel`, reprezentujące dwa różne modele językowe uruchomione lokalnie przez Ollamę.

*   **deepseek\_reasoner\_model:** Ta zmienna przechowuje obiekt modelu, który będzie używany do rozumowania lub logicznego myślenia. Jest on skonfigurowany do korzystania z modelu określonego w `CFG.model1`, czyli `'deepseek-r1:14b'`.  `OpenAIProvider` jest ustawiony na komunikację z serwerem Ollamy działającym lokalnie pod adresem `http://localhost:11434/v1`.

*   **deepseek\_parser\_model:** Ta zmienna przechowuje obiekt modelu, który będzie używany do parsowania lub analizy tekstu. Jest on skonfigurowany do korzystania z modelu określonego w `CFG.model2`, czyli `'qwen2.5'`. Podobnie jak poprzednio, `OpenAIProvider` jest ustawiony na komunikację z lokalnym serwerem Ollamy pod adresem `http://localhost:11434/v1`.

W skrócie, kod ten inicjalizuje dwa modele językowe – DeepSeek-R1 i Qwen2.5 – które będą dostępne do użycia w programie poprzez interfejs udostępniany przez bibliotekę `pydantic_ai`. Oba modele komunikują się z serwerem Ollamy działającym lokalnie.

# Funkcje

In [ ]:
# Define the Joke model
class Joke(BaseModel):
    setup: str = Field(..., description="The setup part of the joke")
    punchline: str = Field(..., description="The punchline of the joke")
    category: str = Field(
        ..., description="Category of the joke (e.g., 'pun', 'wordplay', 'dad joke')"
    )
    thinking: str = Field(..., description="the thought process and planning")


In [ ]:
class JokeAssessment(BaseModel):
    originality: confloat(ge=0, le=10) = Field(
        ..., description="Rating of joke originality from 0-10"
    )
    humor_level: confloat(ge=0, le=10) = Field(
        ..., description="Rating of humor level from 0-10"
    )
    family_friendly: bool = Field(
        ..., description="Whether the joke is family-friendly"
    )
    wordplay_quality: confloat(ge=0, le=10) = Field(
        ..., description="Rating of wordplay quality from 0-10"
    )
    delivery_structure: confloat(ge=0, le=10) = Field(
        ..., description="Rating of setup-punchline structure from 0-10"
    )
    strengths: List[str] = Field(..., description="List of joke's strong points")
    improvement_suggestions: List[str] = Field(
        ...,
        description="Think step buy step and provide a very extensive List of very specific suggestions for improvement",
    )
    overall_score: confloat(ge=0, le=10) = Field(
        ..., description="Overall joke quality score from 0-10"
    )

# Agenci

In [ ]:
joke_creator = Agent(
    model=deepseek_reasoner_model,
    result_type=str,
    deps_type=str,
    system_prompt=(
        "You are a joke creator that specializes in creating funny, family-friendly dad jokes. "
        "Your output MUST follow this EXACT format with EXACT field names:\n\n"
        "Thinking: \n<think>\n[your detailed thought process without changes]\n</think>\n\n"
        "Setup: [setup text]\n"
        "Punchline: [punchline text]\n"
        "Category: [category]\n"
    ),
)

In [ ]:
joke_parser = Agent(
    model=deepseek_parser_model,
    result_type=Joke,
    deps_type=str,  # Takes raw text as input
    system_prompt=(
        "You are a joke parser that extracts components from the input text to create a Joke object. "
        "Your ONLY task is to find and extract these components from the input text:\n"
        "1. The setup (after 'Setup:')\n"
        "2. The punchline (after 'Punchline:')\n"
        "3. The category (after 'Category:')\n"
        "4. The thinking (between <think> tags)\n\n"
        "Return the components in this format:\n"
        "setup=[text after Setup:]\n"
        "punchline=[text after Punchline:]\n"
        "category=[text after Category:]\n"
        "thinking=[text between <think> tags]\n\n"
        "DO NOT generate new content or modify the text in any way.\n"
        "DO NOT include any other text in your response."
    ),
)

In [ ]:
joke_judge = Agent(
    model=deepseek_parser_model,
    result_type=JokeAssessment,
    deps_type=str,
    system_prompt=(
        "You are an expert joke critic who analyzes jokes across multiple dimensions. "
        "You will receive a joke in the format 'Setup: <setup> | Punchline: <punchline>'. "
        "Evaluate the joke and return a JokeAssessment with these fields:"
        "- originality (0-10 float)"
        "- humor_level (0-10 float)"
        "- family_friendly (boolean)"
        "- wordplay_quality (0-10 float)"
        "- delivery_structure (0-10 float)"
        "- strengths (list of strings)"
        "- improvement_suggestions (list of strings)"
        "- overall_score (0-10 float)"
        "\nBe constructive but honest in your feedback."
    ),
)


# Suchar

In [37]:
print("Creating joke...")
joke_text = await joke_creator.run("Create a dad joke")
if not joke_text:
    raise ValueError("Failed to create joke")
print("\nRaw joke text:")
print("-" * 50)
print(joke_text.data)
print("-" * 50)

Creating joke...

Raw joke text:
--------------------------------------------------
<think>
Okay, so the user wants me to create a dad joke following their specific format. First, I need to understand exactly what they're asking for. They provided an example with setup and punchline, and categories like puns, wordplay, etc.

I should focus on family-friendly humor because that's the target audience. Dad jokes usually rely on wordplay or puns, so I'm thinking along those lines. Let me brainstorm a topic—maybe something related to everyday items or animals since those are easy to relate to.

Why don't we ever... oh, wait! Bees might work well. They have a connection with honey and sweetness, which could lead into a pun about being sweetness or desserts.

Putting it together: "Why don’t we ever see bees arguing?" The answer should tie back to the sweet aspect, so something like always agreeing because they’re too busy making honey, implying their "sweet" nature keeps them in harmony.

I t

<ipython-input-37-20e74535158d>:7: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print(joke_text.data)


In [38]:
print("\nParsing joke...")
parsed_joke = await joke_parser.run(joke_text.data)
if not parsed_joke:
    raise ValueError("Failed to parse joke")
print("\nParsed joke text:")
print("-" * 50)
print(parsed_joke.data)
print("-" * 50)



Parsing joke...


<ipython-input-38-11941310037c>:2: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  parsed_joke = await joke_parser.run(joke_text.data)



Parsed joke text:
--------------------------------------------------
setup='Why don’t we ever see bees arguing?' punchline='Because they’re always too busy making honey—agreeable!' category='Puns' thinking='Thinking: I want to create a light-hearted, family-friendly dad joke using wordplay or a pun. Honey is a funny topic because it’s sweet (literally and figuratively). People often associate bees with honey, so it makes sense to connect their behavior to something delicious.'
--------------------------------------------------


<ipython-input-38-11941310037c>:7: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print(parsed_joke.data)


In [ ]:
print("\nAssessing joke...")
assessment = await joke_judge.run(
    f"Setup: {parsed_joke.data.setup} | Punchline: {parsed_joke.data.punchline}"
)
if not assessment:
    raise ValueError("Failed to assess joke")
print("\nAssessment:")
print("-" * 50)
print(assessment.data)
print("-" * 50)



Assessing joke...


<ipython-input-39-b574855aeefd>:3: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  f"Setup: {parsed_joke.data.setup} | Punchline: {parsed_joke.data.punchline}"



Assessment:
--------------------------------------------------
originality=7.5 humor_level=6.5 family_friendly=True wordplay_quality=3.5 delivery_structure=8.0 strengths=['Clever play on words', 'Brief and concise'] improvement_suggestions=['Increase originality by exploring metaphors in other contexts', 'Experiment with a different setup to subvert expectations before the punchline'] overall_score=6.0
--------------------------------------------------


<ipython-input-39-b574855aeefd>:9: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print(assessment.data)


In [ ]:
console = Console()

# Create assessment table
table = Table(title="Joke Assessment", box=box.ROUNDED)
table.add_column("Property", style="cyan", width=20)
table.add_column("Value", style="yellow")

# Add joke content
table.add_row("Setup", parsed_joke.data.setup)
table.add_row("Punchline", parsed_joke.data.punchline)
table.add_row("Category", parsed_joke.data.category)
table.add_row("", "")

# Add scores
scores = assessment.data
table.add_row("Overall Score", f"[bold]{scores.overall_score:.1f}/10")
table.add_row("", "")
table.add_row("Originality", f"{scores.originality:.1f}/10")
table.add_row("Humor Level", f"{scores.humor_level:.1f}/10")
table.add_row("Wordplay", f"{scores.wordplay_quality:.1f}/10")
table.add_row("Structure", f"{scores.delivery_structure:.1f}/10")
table.add_row(
    "Family Friendly",
    "[green]Yes[/green]" if scores.family_friendly else "[red]No[/red]",
)

# Add strengths
table.add_row("", "")
table.add_row("[bold]Strengths", "")
for strength in scores.strengths:
    table.add_row("", f"• {strength}")

# Add improvement suggestions
table.add_row("", "")
table.add_row("[bold]Improvements", "")
for suggestion in scores.improvement_suggestions:
    table.add_row("", f"• {suggestion}")

# Add thinking process
table.add_row("", "")
table.add_row("[bold]Thinking Process", "")
thinking_lines = parsed_joke.data.thinking.split("\n")
for line in thinking_lines:
    if line.strip():
        table.add_row("", line)

# Print the table
console.print("\n")
console.print(table)


<ipython-input-40-299fc290ecef>:9: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  table.add_row("Setup", parsed_joke.data.setup)
<ipython-input-40-299fc290ecef>:10: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  table.add_row("Punchline", parsed_joke.data.punchline)
<ipython-input-40-299fc290ecef>:11: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  table.add_row("Category", parsed_joke.data.category)
<ipython-input-40-299fc290ecef>:15: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  scores = assessment.data
<ipython-input-40-299fc290ecef>:42: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  thinking_lines = parsed_joke.data.thinking.split('\n')


                                                  Joke Assessment                                                  
╭──────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────╮
│ Property             │ Value                                                                                    │
├──────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────┤
│ Setup                │ Why don’t we ever see bees arguing?                                                      │
│ Punchline            │ Because they’re always too busy making honey—agreeable!                                  │
│ Category             │ Puns                                                                                     │
│                      │                                                                                          │
│ Overall Score        │ 6.0/10                                                                                   │
│                      │                                                                                          │
│ Originality          │ 7.5/10                                                                                   │
│ Humor Level          │ 6.5/10                                                                                   │
│ Wordplay             │ 3.5/10                                                                                   │
│ Structure            │ 8.0/10                                                                                   │
│ Family Friendly      │ Yes                                                                                      │
│                      │                                                                                          │
│ Strengths            │                                                                                          │
│                      │ • Clever play on words                                                                   │
│                      │ • Brief and concise                                                                      │
│                      │                                                                                          │
│ Improvements         │                                                                                          │
│                      │ • Increase originality by exploring metaphors in other contexts                          │
│                      │ • Experiment with a different setup to subvert expectations before the punchline         │
│                      │                                                                                          │
│ Thinking Process     │                                                                                          │
│                      │ Thinking: I want to create a light-hearted, family-friendly dad joke using wordplay or a │
│                      │ pun. Honey is a funny topic because it’s sweet (literally and figuratively). People      │
│                      │ often associate bees with honey, so it makes sense to connect their behavior to          │
│                      │ something delicious.                                                                     │
╰──────────────────────┴──────────────────────────────────────────────────────────────────────────────────────────╯

Ten kod tworzy i wyświetla tabelę w konsoli, prezentującą szczegółową ocenę wygenerowanego żartu. Wykorzystuje bibliotekę `rich` do formatowania tekstu i tworzenia atrakcyjnej wizualnie tabeli.

*   **console = Console():** Tworzy obiekt `Console` z biblioteki `rich`, który będzie używany do wyświetlania danych w konsoli.
*   **table = Table(title="Joke Assessment", box=box.ROUNDED):** Tworzy obiekt `Table` z tytułem "Joke Assessment" i zaokrąglonymi krawędziami (styl `box.ROUNDED`).
*   **table.add\_column(...):** Dodaje dwie kolumny do tabeli: "Property" (w kolorze cyjanowym, szerokość 20 znaków) oraz "Value" (w kolorze żółtym).
*   **Dodawanie zawartości żartu:** Kod dodaje wiersze do tabeli zawierające treść żartu: setup, punchline i category.
*   **Dodawanie ocen:** Pobiera dane oceny z obiektu `assessment.data` (który jest instancją klasy `JokeAssessment`) i dodaje wiersze do tabeli prezentujące poszczególne kryteria oceny: overall score, originality, humor level, wordplay quality, structure oraz family-friendly status.  Wartości liczbowe są formatowane do jednej cyfry po przecinku (`:.1f`). Status "family-friendly" jest wyświetlany w kolorze zielonym ("Yes") lub czerwonym ("No").
*   **Dodawanie mocnych stron:** Dodaje nagłówek "Strengths" i następnie dodaje wiersze dla każdej z mocnych stron żartu, poprzedzając je kropką.
*   **Dodawanie sugestii poprawy:** Dodaje nagłówek "Improvements" i następnie dodaje wiersze dla każdej z sugestii dotyczących poprawy żartu, poprzedzając je kropką.
*   **Dodawanie procesu myślowego:** Dodaje nagłówek "Thinking Process" i następnie dodaje wiersze zawierające poszczególne linie procesu myślowego agenta (pobrane z `parsed_joke.data.thinking`), dzieląc go na linie za pomocą `\n`.
*   **console.print("\n"):** Dodaje pustą linię przed wyświetleniem tabeli.
*   **console.print(table):** Wyświetla utworzoną tabelę w konsoli, wykorzystując obiekt `Console` do formatowania i kolorowania tekstu.

W skrócie, kod ten tworzy czytelną i estetyczną prezentację oceny żartu w formie tabeli, zawierającą zarówno treść żartu, jak i szczegółową analizę jego jakości oraz sugestie dotyczące poprawy.

# Suchar 2.0

In [ ]:
print("1. Creating new joke variation...")
improvement_prompt = (
    "Create an improved variation of this dad joke using ALL the information below \n"
    + "Do not change the Setup \n"
    + f"Setup: {parsed_joke.data.setup}\n"
    + f"Punchline: {parsed_joke.data.punchline}\n"
    + f"Category: {parsed_joke.data.category}\n"
    + f"Thinking: {parsed_joke.data.thinking}\n"
    + f"Scores: {scores}"
)

print("Improved prompt:")
print("-" * 50)
print(improvement_prompt)
print("-" * 50)


joke_text = await joke_creator.run(improvement_prompt)

if not joke_text:
    raise ValueError("Failed to create new joke")

print("\nNew joke text:")
print("-" * 50)
print(joke_text.data)
print("-" * 50)


1. Creating new joke variation...
Improved prompt:
--------------------------------------------------
Create an improved variation of this dad joke using ALL the information below 
Do not change the Setup 
Setup: Why don’t we ever see bees arguing?
Punchline: Because they’re always too busy making honey—agreeable!
Category: Puns
Thinking: Thinking: I want to create a light-hearted, family-friendly dad joke using wordplay or a pun. Honey is a funny topic because it’s sweet (literally and figuratively). People often associate bees with honey, so it makes sense to connect their behavior to something delicious.
Scores: originality=7.5 humor_level=6.5 family_friendly=True wordplay_quality=3.5 delivery_structure=8.0 strengths=['Clever play on words', 'Brief and concise'] improvement_suggestions=['Increase originality by exploring metaphors in other contexts', 'Experiment with a different setup to subvert expectations before the punchline'] overall_score=6.0
----------------------------------

<ipython-input-41-af505e6ae695>:5: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  f"Setup: {parsed_joke.data.setup}\n" +
<ipython-input-41-af505e6ae695>:6: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  f"Punchline: {parsed_joke.data.punchline}\n" +
<ipython-input-41-af505e6ae695>:7: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  f"Category: {parsed_joke.data.category}\n" +
<ipython-input-41-af505e6ae695>:8: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  f"Thinking: {parsed_joke.data.thinking}\n" +



New joke text:
--------------------------------------------------
<think>
Alright, so I have this user who wants me to create a joke about bees and honey. The setup is already given: "Why don’t we ever see bees arguing?" And the punchline is "Because they’re always too busy making honey—agreeable!" with the category listed as Puns.

Hmm, first off, I need to understand what makes this joke work. It's a pun on "busy" and "honey," combining them in a clever way. The bee's job is to make honey, which ties into them being agreeable because they're too occupied with their work. That's a nice wordplay.

Now, thinking about family-friendly dad jokes, they usually rely on puns or wordplay that's easy to understand but still makes you laugh. This joke fits well in that category because it's simple and funny without any adult themes.

I should consider if this joke is original enough. Bees making honey is such a common topic, so how can I make it fresh? Maybe play with the "busy" aspect more or

<ipython-input-41-af505e6ae695>:25: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print(joke_text.data)


In [ ]:
print("\nParsing joke...")
parsed_joke = await joke_parser.run(joke_text.data)
if not parsed_joke:
    raise ValueError("Failed to parse joke")
print("\nParsed joke text:")
print("-" * 50)
print(parsed_joke.data)
print("-" * 50)



Parsing joke...


<ipython-input-42-1c6b88054d33>:2: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  parsed_joke = await joke_parser.run(joke_text.data)



Parsed joke text:
--------------------------------------------------
setup='Why don’t we ever see bees arguing?' punchline='Because they’re always too busy making honey—agreeable!' category='Puns' thinking="Alright, so I have this user who wants me to create a joke about bees and honey. The setup is already given: 'Why don’t we ever see bees arguing?' And the punchline is 'Because they’re always too busy making honey—agreeable!' with the category listed as Puns.\n\nHmm, first off, I need to understand what makes this joke work. It's a pun on 'busy' and 'honey,' combining them in a clever way. The bee's job is to make honey, which ties into them being agreeable because they're too occupied with their work. That's a nice wordplay.\n\nNow, thinking about family-friendly dad jokes, they usually rely on puns or wordplay that's easy to understand but still makes you laugh. This joke fits well in that category because it's simple and funny without any adult themes.\n\nThe humor level here is

<ipython-input-42-1c6b88054d33>:7: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print(parsed_joke.data)


In [43]:
print("\nAssessing joke...")
assessment = await joke_judge.run(
    f"Setup: {parsed_joke.data.setup} | Punchline: {parsed_joke.data.punchline}"
)
if not assessment:
    raise ValueError("Failed to assess joke")
print("\nAssessment:")
print("-" * 50)
print(assessment.data)
print("-" * 50)



Assessing joke...


<ipython-input-43-2c9fcb1daa0b>:3: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  f"Setup: {parsed_joke.data.setup} | Punchline: {parsed_joke.data.punchline}"



Assessment:
--------------------------------------------------
originality=5.5 humor_level=6.0 family_friendly=True wordplay_quality=7.5 delivery_structure=8.0 strengths=['clever play on words', 'positive outlook'] improvement_suggestions=['Consider refining the punchline for smoother delivery', 'Explore more unique scenarios to increase originality'] overall_score=6.3
--------------------------------------------------


<ipython-input-43-2c9fcb1daa0b>:9: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print(assessment.data)


In [ ]:
console = Console()

# Create assessment table
table = Table(title="Joke Assessment", box=box.ROUNDED)
table.add_column("Property", style="cyan", width=20)
table.add_column("Value", style="yellow")

# Add joke content
table.add_row("Setup", parsed_joke.data.setup)
table.add_row("Punchline", parsed_joke.data.punchline)
table.add_row("Category", parsed_joke.data.category)
table.add_row("", "")

# Add scores
scores = assessment.data
table.add_row("Overall Score", f"[bold]{scores.overall_score:.1f}/10")
table.add_row("", "")
table.add_row("Originality", f"{scores.originality:.1f}/10")
table.add_row("Humor Level", f"{scores.humor_level:.1f}/10")
table.add_row("Wordplay", f"{scores.wordplay_quality:.1f}/10")
table.add_row("Structure", f"{scores.delivery_structure:.1f}/10")
table.add_row(
    "Family Friendly",
    "[green]Yes[/green]" if scores.family_friendly else "[red]No[/red]",
)

# Add strengths
table.add_row("", "")
table.add_row("[bold]Strengths", "")
for strength in scores.strengths:
    table.add_row("", f"• {strength}")

# Add improvement suggestions
table.add_row("", "")
table.add_row("[bold]Improvements", "")
for suggestion in scores.improvement_suggestions:
    table.add_row("", f"• {suggestion}")

# Add thinking process
table.add_row("", "")
table.add_row("[bold]Thinking Process", "")
thinking_lines = parsed_joke.data.thinking.split("\n")
for line in thinking_lines:
    if line.strip():
        table.add_row("", line)

# Print the table
console.print("\n")
console.print(table)


<ipython-input-44-299fc290ecef>:9: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  table.add_row("Setup", parsed_joke.data.setup)
<ipython-input-44-299fc290ecef>:10: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  table.add_row("Punchline", parsed_joke.data.punchline)
<ipython-input-44-299fc290ecef>:11: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  table.add_row("Category", parsed_joke.data.category)
<ipython-input-44-299fc290ecef>:15: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  scores = assessment.data
<ipython-input-44-299fc290ecef>:42: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  thinking_lines = parsed_joke.data.thinking.split('\n')


                                                  Joke Assessment                                                  
╭──────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────╮
│ Property             │ Value                                                                                    │
├──────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────┤
│ Setup                │ Why don’t we ever see bees arguing?                                                      │
│ Punchline            │ Because they’re always too busy making honey—agreeable!                                  │
│ Category             │ Puns                                                                                     │
│                      │                                                                                          │
│ Overall Score        │ 6.3/10                                                                                   │
│                      │                                                                                          │
│ Originality          │ 5.5/10                                                                                   │
│ Humor Level          │ 6.0/10                                                                                   │
│ Wordplay             │ 7.5/10                                                                                   │
│ Structure            │ 8.0/10                                                                                   │
│ Family Friendly      │ Yes                                                                                      │
│                      │                                                                                          │
│ Strengths            │                                                                                          │
│                      │ • clever play on words                                                                   │
│                      │ • positive outlook                                                                       │
│                      │                                                                                          │
│ Improvements         │                                                                                          │
│                      │ • Consider refining the punchline for smoother delivery                                  │
│                      │ • Explore more unique scenarios to increase originality                                  │
│                      │                                                                                          │
│ Thinking Process     │                                                                                          │
│                      │ Alright, so I have this user who wants me to create a joke about bees and honey. The     │
│                      │ setup is already given: 'Why don’t we ever see bees arguing?' And the punchline is       │
│                      │ 'Because they’re always too busy making honey—agreeable!' with the category listed as    │
│                      │ Puns.                                                                                    │
│                      │ Hmm, first off, I need to understand what makes this joke work. It's a pun on 'busy' and │
│                      │ 'honey,' combining them in a clever way. The bee's job is to make honey, which ties into │
│                      │ them being agreeable because they're too occupied with their work. That's a nice         │
│                      │ wordplay.                                                                                │
│                      │ Now, thinking about family-friendly dad jokes, they usually rely on puns or wordplay     │
│                      │ that's easy to understand but s